<a href="https://colab.research.google.com/github/tharujayasinghe163/Statistical-Learning-e22163/blob/main/Data_wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

# ==========================================================
# MODULAR DATA SANITIZATION & EXPLORATION ENGINE
# ==========================================================

# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import io
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

from google.colab import files

from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    OneHotEncoder,
    OrdinalEncoder
)

from scipy.stats import chi2_contingency
from scipy.stats import pointbiserialr
from scipy.stats import f_oneway

from IPython.display import display, HTML


# ==========================================================
# DATA INSPECTOR CLASS
# ==========================================================

class DataInspector:
    """
    Advanced reusable toolkit for:
    - Data Cleaning
    - Data Exploration
    - Feature Engineering
    - Statistical Analysis
    - Interactive Visualization
    """

    def __init__(self):

        self.df = None
        self.numeric_cols = []
        self.categorical_cols = []

    # ======================================================
    # 1. DATA INGESTION & SANITIZATION
    # ======================================================

    def upload_data(self):

        """
        Upload CSV file directly in Google Colab.
        """

        print("Upload CSV File")

        uploaded = files.upload()

        if not uploaded:
            print("No file uploaded.")
            return

        filename = list(uploaded.keys())[0]

        garbage_strings = [
            '?',
            'n/a',
            'N/A',
            'NULL',
            'null',
            ' ',
            ''
        ]

        self.df = pd.read_csv(
            io.BytesIO(uploaded[filename]),
            na_values=garbage_strings
        )

        print(f"{filename} loaded successfully.")

        self._auto_type_correction()

    def _auto_type_correction(self):

        """
        Automatically converts columns to numeric
        if possible.
        """

        if self.df is None:
            return

        for col in self.df.columns:

            converted = pd.to_numeric(
                self.df[col],
                errors='coerce'
            )

            if not converted.isna().all():
                self.df[col] = converted

        self.numeric_cols = list(
            self.df.select_dtypes(
                include=np.number
            ).columns
        )

        self.categorical_cols = list(
            self.df.select_dtypes(
                exclude=np.number
            ).columns
        )

    # ======================================================
    # 2. DATA SUMMARY
    # ======================================================

    def get_summary(self):

        """
        Prints complete dataset structure.
        """

        if self.df is None:
            print("Dataset not loaded.")
            return

        print("=" * 60)
        print("DATASET SUMMARY")
        print("=" * 60)

        print(f"Rows    : {self.df.shape[0]}")
        print(f"Columns : {self.df.shape[1]}")

        print("\nNUMERICAL COLUMNS")
        print(self.numeric_cols)

        print("\nCATEGORICAL COLUMNS")
        print(self.categorical_cols)

        print("\nMISSING VALUES")
        print(self.df.isnull().sum())

        print("\nFIRST 20 ROWS")
        display(self.df.head(20))

    # ======================================================
    # 3. HANDLE MISSING VALUES
    # ======================================================

    def handle_missing_values(
        self,
        strategy='median',
        fill_value=None
    ):

        """
        Missing value imputation.
        """

        if self.df is None:
            return

        for col in self.df.columns:

            if self.df[col].isna().sum() == 0:
                continue

            # Numerical Columns
            if col in self.numeric_cols:

                if strategy == 'mean':

                    self.df[col].fillna(
                        self.df[col].mean(),
                        inplace=True
                    )

                elif strategy == 'median':

                    self.df[col].fillna(
                        self.df[col].median(),
                        inplace=True
                    )

                elif strategy == 'mode':

                    self.df[col].fillna(
                        self.df[col].mode()[0],
                        inplace=True
                    )

                elif strategy == 'constant':

                    self.df[col].fillna(
                        fill_value,
                        inplace=True
                    )

            # Categorical Columns
            else:

                if strategy == 'constant':

                    self.df[col].fillna(
                        fill_value,
                        inplace=True
                    )

                else:

                    self.df[col].fillna(
                        self.df[col].mode()[0],
                        inplace=True
                    )

        print(f"Missing values handled using '{strategy}' strategy.")

    # ======================================================
    # 4. DUPLICATE REMOVAL
    # ======================================================

    def remove_duplicates(self):

        """
        Removes duplicate rows.
        """

        if self.df is None:
            return

        before = self.df.shape[0]

        self.df.drop_duplicates(inplace=True)

        after = self.df.shape[0]

        print(f"Removed {before - after} duplicate rows.")

    # ======================================================
    # 5. OUTLIER HANDLING
    # ======================================================

    def handle_outliers(
        self,
        columns=None,
        remove=False
    ):

        """
        Detects/removes outliers using IQR.
        """

        if self.df is None:
            return

        cols = columns if columns else self.numeric_cols

        rows_to_drop = set()

        for col in cols:

            Q1 = self.df[col].quantile(0.25)
            Q3 = self.df[col].quantile(0.75)

            IQR = Q3 - Q1

            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR

            outliers = self.df[
                (self.df[col] < lower) |
                (self.df[col] > upper)
            ]

            print(
                f"{col} -> {len(outliers)} outliers"
            )

            if remove:
                rows_to_drop.update(outliers.index)

        if remove:

            self.df.drop(
                index=list(rows_to_drop),
                inplace=True
            )

            print(
                f"{len(rows_to_drop)} rows removed."
            )

    # ======================================================
    # 6. DELETE ROWS / COLUMNS
    # ======================================================

    def delete_columns(self):

        """
        Delete columns interactively.
        """

        cols = input(
            "Enter columns to delete: "
        )

        cols = [
            c.strip()
            for c in cols.split(',')
            if c.strip() in self.df.columns
        ]

        self.df.drop(columns=cols, inplace=True)

        self._auto_type_correction()

        print("Columns deleted.")

    def delete_rows(self):

        """
        Delete rows interactively.
        """

        rows = input(
            "Enter row indices to delete: "
        )

        rows = [
            int(r.strip())
            for r in rows.split(',')
            if r.strip().isdigit()
        ]

        self.df.drop(index=rows, inplace=True)

        print("Rows deleted.")

    # ======================================================
    # 7. NUMERICAL NORMALIZATION
    # ======================================================

    def extract_normalized_numeric_data(
        self,
        method='standard'
    ):

        """
        Scales numeric columns.
        """

        if not self.numeric_cols:
            return pd.DataFrame()

        if method == 'minmax':

            scaler = MinMaxScaler()

        elif method == 'robust':

            scaler = RobustScaler()

        else:

            scaler = StandardScaler()

        scaled = scaler.fit_transform(
            self.df[self.numeric_cols]
        )

        return pd.DataFrame(
            scaled,
            columns=self.numeric_cols,
            index=self.df.index
        )

    # ======================================================
    # 8. CATEGORICAL ENCODING
    # ======================================================

    def extract_normalized_categorical_data(
        self,
        method='onehot'
    ):

        """
        Encodes categorical features.
        """

        if not self.categorical_cols:
            return pd.DataFrame()

        # ONE HOT
        if method == 'onehot':

            encoder = OneHotEncoder(
                sparse_output=False,
                drop='first'
            )

            encoded = encoder.fit_transform(
                self.df[self.categorical_cols]
            )

            cols = encoder.get_feature_names_out(
                self.categorical_cols
            )

            return pd.DataFrame(
                encoded,
                columns=cols,
                index=self.df.index
            )

        # ORDINAL
        elif method == 'ordinal':

            encoder = OrdinalEncoder()

            encoded = encoder.fit_transform(
                self.df[self.categorical_cols]
            )

            return pd.DataFrame(
                encoded,
                columns=self.categorical_cols,
                index=self.df.index
            )

        # UNIFORM
        elif method == 'uniform':

            encoder = OrdinalEncoder()

            encoded = encoder.fit_transform(
                self.df[self.categorical_cols]
            )

            encoded = (
                encoded - encoded.min(axis=0)
            ) / (
                encoded.max(axis=0) -
                encoded.min(axis=0) + 1e-9
            )

            return pd.DataFrame(
                encoded,
                columns=self.categorical_cols,
                index=self.df.index
            )

    # ======================================================
    # 9. MERGE DATA
    # ======================================================

    def create_normalized_data_df(
        self,
        numeric_method='standard',
        categorical_method='onehot'
    ):

        """
        Creates ML-ready merged dataset.
        """

        num_df = self.extract_normalized_numeric_data(
            method=numeric_method
        )

        cat_df = self.extract_normalized_categorical_data(
            method=categorical_method
        )

        return pd.concat(
            [num_df, cat_df],
            axis=1
        )

    # ======================================================
    # 10. UNIVARIATE PLOTS
    # ======================================================

    def plot_numerical(self, columns):

        """
        Creates violin, scatter,
        and histogram subplots.
        """

        for col in columns:

            if col not in self.numeric_cols:
                continue

            fig = make_subplots(
                rows=1,
                cols=3,
                subplot_titles=[
                    "Violin Plot",
                    "Scatter Plot",
                    "Histogram"
                ]
            )

            fig.add_trace(
                go.Violin(
                    x=self.df[col],
                    box_visible=True,
                    points='all'
                ),
                row=1,
                col=1
            )

            fig.add_trace(
                go.Scatter(
                    y=self.df[col],
                    mode='markers'
                ),
                row=1,
                col=2
            )

            fig.add_trace(
                go.Histogram(
                    x=self.df[col]
                ),
                row=1,
                col=3
            )

            fig.update_layout(
                title=f"Distribution of {col}",
                height=400,
                showlegend=False
            )

            fig.show()

    # ======================================================
    # 11. SMART RELATIONSHIP PLOTS
    # ======================================================

    def plot_relationship(self, var1, var2):

        """
        Automatically selects chart type.
        """

        v1_num = var1 in self.numeric_cols
        v2_num = var2 in self.numeric_cols

        # Num-Num
        if v1_num and v2_num:

            fig = px.scatter(
                self.df,
                x=var1,
                y=var2,
                trendline='ols',
                title=f"{var1} vs {var2}"
            )

        # Cat-Cat
        elif not v1_num and not v2_num:

            fig = px.histogram(
                self.df,
                x=var1,
                color=var2,
                barmode='group',
                title=f"{var1} vs {var2}"
            )

        # Mixed
        else:

            cat_var = var1 if not v1_num else var2
            num_var = var2 if not v1_num else var1

            fig = px.box(
                self.df,
                x=cat_var,
                y=num_var,
                points='all',
                title=f"{num_var} by {cat_var}"
            )

        fig.show()

    # ======================================================
    # 12. ASSOCIATION HEATMAP
    # ======================================================

    def cramers_v(self, x, y):

        """
        Computes Cramer's V.
        """

        confusion = pd.crosstab(x, y)

        chi2 = chi2_contingency(confusion)[0]

        n = confusion.sum().sum()

        phi2 = chi2 / n

        r, k = confusion.shape

        return np.sqrt(
            phi2 / min(k - 1, r - 1)
        )

    def plot_all_associations_heatmap(self):

        """
        Generates unified statistical heatmap.
        """

        cols = self.df.columns

        matrix = pd.DataFrame(
            np.zeros((len(cols), len(cols))),
            columns=cols,
            index=cols
        )

        for c1 in cols:
            for c2 in cols:

                try:

                    # Num-Num
                    if (
                        c1 in self.numeric_cols and
                        c2 in self.numeric_cols
                    ):

                        val = self.df[c1].corr(
                            self.df[c2]
                        )

                    # Cat-Cat
                    elif (
                        c1 in self.categorical_cols and
                        c2 in self.categorical_cols
                    ):

                        val = self.cramers_v(
                            self.df[c1],
                            self.df[c2]
                        )

                    # Mixed
                    else:

                        num_col = (
                            c1 if c1 in self.numeric_cols
                            else c2
                        )

                        cat_col = (
                            c2 if c1 in self.numeric_cols
                            else c1
                        )

                        codes = self.df[
                            cat_col
                        ].astype(
                            'category'
                        ).cat.codes

                        val = self.df[
                            num_col
                        ].corr(codes)

                    matrix.loc[c1, c2] = val

                except:
                    matrix.loc[c1, c2] = 0

        fig = px.imshow(
            matrix,
            text_auto='.2f',
            color_continuous_scale='RdBu_r',
            title='Unified Association Heatmap'
        )

        fig.show()


# ==========================================================
# MODULAR PLOTTING CLASS
# ==========================================================

class PlottingMethods:

    """
    Reusable modular plotting methods.
    """

    def plot_bar_chart(
        self,
        x,
        y,
        data,
        title=None
    ):

        fig = px.bar(
            data,
            x=x,
            y=y,
            title=title,
            text_auto=True
        )

        return {
            "status": "success",
            "html": fig.to_html(
                include_plotlyjs='cdn',
                full_html=False
            )
        }

    def plot_pie_chart(
        self,
        names,
        values,
        data,
        title=None
    ):

        fig = px.pie(
            data,
            names=names,
            values=values,
            hole=0.4,
            title=title
        )

        return {
            "status": "success",
            "html": fig.to_html(
                include_plotlyjs='cdn',
                full_html=False
            )
        }

    def plot_histogram(
        self,
        x,
        data,
        title=None
    ):

        fig = px.histogram(
            data,
            x=x,
            title=title
        )

        return {
            "status": "success",
            "html": fig.to_html(
                include_plotlyjs='cdn',
                full_html=False
            )
        }

    def display_image(self, result):

        """
        Displays generated HTML chart.
        """

        if result["status"] == "success":

            display(
                HTML(result["html"])
            )

        else:

            print(result["message"])


# ==========================================================
# TESTING USING TITANIC DATASET
# ==========================================================

inspector = DataInspector()

plotter = PlottingMethods()

# Load Titanic Dataset
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

inspector.df = pd.read_csv(url)

inspector._auto_type_correction()

# Summary
inspector.get_summary()

# Missing Values
inspector.handle_missing_values(
    strategy='median'
)

# Remove Duplicates
inspector.remove_duplicates()
'''python
# Outlier Detection
inspector.handle_outliers(
    columns=['Age', 'Fare'],
    remove=False
)

# ML Ready Dataset
ml_ready_df = inspector.create_normalized_data_df(
    numeric_method='robust',
    categorical_method='onehot'
)

print("\nML READY DATASET")
display(ml_ready_df.head())

# Visualization
inspector.plot_numerical(
    ['Age', 'Fare']
)

inspector.plot_relationship(
    'Pclass',
    'Fare'
)

# Heatmap
inspector.plot_all_associations_heatmap()

# Custom Plotting Class
result = plotter.plot_pie_chart(
    names='Sex',
    values='PassengerId',
    data=inspector.df,
    title='Gender Distribution'
)

plotter.display_image(result)
'''


DATASET SUMMARY
Rows    : 891
Columns : 12

NUMERICAL COLUMNS
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']

CATEGORICAL COLUMNS
['Name', 'Sex', 'Cabin', 'Embarked']

MISSING VALUES
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket         230
Fare             0
Cabin          687
Embarked         2
dtype: int64

FIRST 20 ROWS


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


Missing values handled using 'median' strategy.
Removed 0 duplicate rows.


/tmp/ipykernel_3631/3052338804.py:193: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



/tmp/ipykernel_3631/3052338804.py:224: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)

'python\n# Outlier Detection\ninspector.handle_outliers(\n    columns=[\'Age\', \'Fare\'],\n    remove=False\n)\n\n# ML Ready Dataset\nml_ready_df = inspector.create_normalized_data_df(\n    numeric_method=\'robust\',\n    categorical_method=\'onehot\'\n)\n\nprint("\nML READY DATASET")\ndisplay(ml_ready_df.head())\n\n# Visualization\ninspector.plot_numerical(\n    [\'Age\', \'Fare\']\n)\n\ninspector.plot_relationship(\n    \'Pclass\',\n    \'Fare\'\n)\n\n# Heatmap\ninspector.plot_all_associations_heatmap()\n\n# Custom Plotting Class\nresult = plotter.plot_pie_chart(\n    names=\'Sex\',\n    values=\'PassengerId\',\n    data=inspector.df,\n    title=\'Gender Distribution\'\n)\n\nplotter.display_image(result)\n'